In [2]:
import cv2
import numpy as np

image = cv2.imread(r'E:\Computer Vision\LAB\lab07\calibration_wide\pattern.png')
image = cv2.resize(image, (1280, 720))
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
checkerboard_size = (9, 6)
ret, corners = cv2.findChessboardCorners(gray, checkerboard_size, None)

if ret:
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 
                30, 0.001)

    refined_corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)

    cv2.drawChessboardCorners(image, checkerboard_size, refined_corners, ret)

    cv2.imshow('Detected Corners', image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Checkerboard corners not detected")

In [3]:
import cv2
import numpy as np

# Load image
image = cv2.imread(r'E:\Computer Vision\LAB\lab07\calibration_wide\pattern.png')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Checkerboard size (inner corners)
checkerboard_size = (9, 6)

# Square size in mm
square_size = 30

# Prepare object points (3D points in real world)
objp = np.zeros((checkerboard_size[0] * checkerboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:checkerboard_size[0], 0:checkerboard_size[1]].T.reshape(-1, 2)
objp = objp * square_size

# Arrays to store points
objpoints = []  # 3D points
imgpoints = []  # 2D points

# Detect corners
ret, corners = cv2.findChessboardCorners(gray, checkerboard_size, None)

if ret:
    # Refine corners
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)

    objpoints.append(objp)
    imgpoints.append(corners2)

    # Perform calibration
    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, gray.shape[::-1], None, None
    )

    print("Camera Matrix (K):\n", K)
    print("\nDistortion Coefficients:\n", dist)

else:
    print("Corners not detected")

Camera Matrix (K):
 [[7.86145384e+04 0.00000000e+00 8.76507972e+02]
 [0.00000000e+00 7.85922994e+04 6.19353719e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Distortion Coefficients:
 [[ 1.26358145e-01  5.74438558e-04  1.29755172e-06 -6.14166790e-03
   6.78508977e-08]]


In [ ]:
import cv2
import numpy as np
import glob

# Checkerboard settings
checkerboard_size = (9, 6)   # inner corners
square_size = 30             # mm

# Prepare 3D object points
objp = np.zeros((checkerboard_size[0] * checkerboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:checkerboard_size[0], 0:checkerboard_size[1]].T.reshape(-1, 2)
objp *= square_size

# Arrays for calibration
objpoints = []
imgpoints = []

# Load all calibration images
images = glob.glob(r'E:\Computer Vision\LAB\lab07\calibration_wide\*.png')

for fname in images:
    image = cv2.imread(fname)

    if image is None:
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(
        gray,
        checkerboard_size,
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if ret:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

        corners2 = cv2.cornerSubPix(
            gray, corners, (11, 11), (-1, -1), criteria
        )

        objpoints.append(objp)
        imgpoints.append(corners2)

# Camera calibration
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

print("Improved Camera Matrix:\n", K)
print("\nDistortion Coefficients:\n", dist)

# Select one image for undistortion
img = cv2.imread(images[0])
h, w = img.shape[:2]

# Compute optimal new camera matrix
new_K, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))

# Undistort image
undistorted = cv2.undistort(img, K, dist, None, new_K)

# Crop valid region
x, y, w, h = roi
undistorted = undistorted[y:y+h, x:x+w]

# Show comparison
cv2.imshow("Original Image", img)
cv2.imshow("Undistorted Image", undistorted)
cv2.waitKey(0)
cv2.destroyAllWindows()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\calib3d\src\calibration.cpp:1382: error: (-215:Assertion failed) nimages > 0 in function 'cv::calibrateCameraRO'
